# 05 · LSTM forecasts of volatility and volume, and evaluation

**Question.** Does text sentiment help predict **next-day volatility** and **next-day trading volume** for GME, AMC and BB, beyond what past prices and volumes already tell us?

**Data.** `panel_daily.csv` from 04 (one row per ticker × trading day, 2010–2023).

**Targets (day t+1).** `y_log_park` = log Parkinson range volatility; `y_abn_logvol` = log volume minus its 20-day mean.

**Inputs (window of the last 20 trading days).**
* *market*: return, |return|, log range vol, 5/20-day vol, abnormal volume, gaps to the 5/20/50-day moving averages, MA5/MA20 cross, ticker dummies
* *+ news (BiLSTM)*: count, sentiment, clarity, ambiguity, QISS from 03b
* *+ news (LLM)*: the same metrics computed from the LLM teacher labels

**Models.** naive (tomorrow = today), ridge regression on day-t features, and an LSTM(32) on 20-day sequences (3 seeds, averaged). Every model is scored on the same rows.

**Splits.** train 2010–2018 · validation 2019–2020 (early stopping) · test **2021** (meme squeeze) and **2022–23**, reported separately.

**Metric.** out-of-sample R² vs the training-mean forecast, plus Diebold–Mariano tests between models.

**Reddit.** Reddit data exist only for Dec 2020 – Jul 2021, which is too short to train an LSTM. Section 4 tests it with a pooled regression on 2021 instead.

In [2]:
# 0. Setup
from google.colab import drive
drive.mount('/content/drive', timeout_ms=600000)   # long timeout: leaves time to approve Drive access
import os, glob, time, random, numpy as np, pandas as pd
import torch, torch.nn as nn
from scipy import stats
import statsmodels.api as sm

def find_project_dir(name="dl-quant_project"):
    cands = [f"/content/drive/MyDrive/Colab Notebooks/{name}", f"/content/drive/MyDrive/{name}",
             *glob.glob(f"/content/drive/MyDrive/*/{name}"), *glob.glob(f"/content/drive/Shareddrives/*/{name}")]
    for p in cands:
        if os.path.isdir(p): return p
    raise FileNotFoundError("Add a shortcut to the shared 'dl-quant_project' folder in My Drive (see 00_START_HERE).")
DATA_DIR = os.path.join(find_project_dir(), "DataSource")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.width", 200)
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

SENT_VERSION = "v2_reddit"   # must match the panel built in 04 ("v1" = 03b BiLSTM, "v2_reddit" = 03d BiLSTM)
D = pd.read_csv(os.path.join(DATA_DIR, f"panel_daily_{SENT_VERSION}.csv" if SENT_VERSION != "v1" else "panel_daily.csv"), parse_dates=["date"])
D = D[D.year >= 2010].sort_values(["ticker", "date"]).reset_index(drop=True)
MARKET = ["ret", "abs_ret", "log_park", "vol_5", "vol_20", "abn_logvol", "ma5_gap", "ma20_gap", "ma50_gap", "ma5_20_cross"]
TEXT_B = ["nb_has", "nb_logn", "nb_sent", "nb_clar", "nb_amb", "qiss_news"]
TEXT_L = ["nl_has", "nl_logn", "nl_sent", "nl_clar", "nl_amb", "qiss_news_llm"]
TICK = ["is_GME", "is_AMC", "is_BB"]
for t in ["GME", "AMC", "BB"]: D[f"is_{t}"] = (D.ticker == t).astype(float)
D[["qiss_news", "qiss_news_llm"]] = D[["qiss_news", "qiss_news_llm"]].fillna(0.0)
TARGETS = {"y_log_park": "next-day log range volatility", "y_abn_logvol": "next-day abnormal log volume"}
print("device:", DEVICE); print(D.groupby("split").size())

Mounted at /content/drive
device: cpu
split
test_2021        756
test_2022_23    1503
train           5795
val             1515
dtype: int64


## 1. Sequences and the LSTM

In [3]:
WIN = 20
def make_sequences(feats, target):
    tr = D.split == "train"
    mu = D.loc[tr, feats].mean(); sd = D.loc[tr, feats].std().replace(0, 1)
    Z = ((D[feats] - mu) / sd).clip(-10, 10).fillna(0.0).to_numpy(np.float32)     # scaled with TRAIN statistics only
    ok_row = D[feats].notna().all(1).to_numpy() & D[target].notna().to_numpy()
    y = D[target].to_numpy(np.float32)
    Xs, ys, idx = [], [], []
    for t, g in D.groupby("ticker"):
        ii = g.index.to_numpy()
        for k in range(WIN - 1, len(ii)):
            j = ii[k]
            if ok_row[j] and ok_row[ii[k - WIN + 1]]:
                Xs.append(Z[ii[k - WIN + 1:k + 1]]); ys.append(y[j]); idx.append(j)
    return np.stack(Xs), np.array(ys, np.float32), np.array(idx)

class SeqLSTM(nn.Module):
    def __init__(self, n_in, hidden=32, drop=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_in, hidden, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(hidden, 16), nn.ReLU(), nn.Linear(16, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1]).squeeze(-1)

def fit_lstm(X, y, idx, seed, epochs=60, patience=8, bs=128, lr=1e-3):
    set_seed(seed)
    sp = D.split.to_numpy()[idx]
    tr, va = sp == "train", sp == "val"
    ym, ys_ = y[tr].mean(), y[tr].std()
    Xt = torch.tensor(X); yt = torch.tensor((y - ym) / ys_)
    model = SeqLSTM(X.shape[2]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    tri = np.where(tr)[0]; vai = np.where(va)[0]
    best, state, bad = np.inf, None, 0
    for ep in range(epochs):
        model.train(); perm = np.random.permutation(tri)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            loss = ((model(Xt[b].to(DEVICE)) - yt[b].to(DEVICE)) ** 2).mean()
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval()
        with torch.no_grad(): vl = ((model(Xt[vai].to(DEVICE)) - yt[vai].to(DEVICE)) ** 2).mean().item()
        if vl < best - 1e-5: best, bad, state = vl, 0, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state); model.eval()
    with torch.no_grad(): pred = torch.cat([model(Xt[i:i + 2048].to(DEVICE)).cpu() for i in range(0, len(Xt), 2048)]).numpy()
    return pred * ys_ + ym, ep + 1

## 2. Train and compare all models

In [4]:
def fit_linear(feats, target, idx):
    rows = D.loc[idx]
    tr = (rows.split == "train").to_numpy()
    X = rows[feats].fillna(0.0).to_numpy(); y = rows[target].to_numpy()
    mu, sd = X[tr].mean(0), X[tr].std(0); sd[sd == 0] = 1
    Xz = np.c_[np.ones(len(X)), np.clip((X - mu) / sd, -10, 10)]
    lam = 1.0 * np.eye(Xz.shape[1]); lam[0, 0] = 0
    beta = np.linalg.solve(Xz[tr].T @ Xz[tr] + lam, Xz[tr].T @ y[tr])
    return Xz @ beta

def dm_test(e1, e2, lag=5):
    d = e1 ** 2 - e2 ** 2
    T = len(d); dbar = d.mean(); dc = d - dbar
    gamma = [dc[k:] @ dc[:T - k] / T for k in range(lag + 1)]
    var = gamma[0] + 2 * sum((1 - k / (lag + 1)) * gamma[k] for k in range(1, lag + 1))
    stat = dbar / np.sqrt(var / T)
    return stat, 2 * (1 - stats.norm.cdf(abs(stat)))

SEEDS = [42, 43, 44]
FEATURE_SETS = {"market": MARKET + TICK, "market + news (BiLSTM)": MARKET + TICK + TEXT_B, "market + news (LLM)": MARKET + TICK + TEXT_L}
results, preds = [], {}
for target in TARGETS:
    _, _, idx_common = make_sequences(FEATURE_SETS["market + news (BiLSTM)"], target)   # identical rows for all models
    rows = D.loc[idx_common]
    y = rows[target].to_numpy(); sp = rows.split.to_numpy(); ytr_mean = y[sp == "train"].mean()
    P_ = {"naive (today)": rows["log_park"].to_numpy() if target == "y_log_park" else rows["abn_logvol"].to_numpy()}
    for name, feats in FEATURE_SETS.items():
        P_[f"ridge: {name}"] = fit_linear(feats, target, idx_common)
        t0 = time.time(); ps = []
        X, yy, idx = make_sequences(feats, target)
        assert (idx == idx_common).all()
        for s in SEEDS:
            p, ep = fit_lstm(X, yy, idx, s); ps.append(p)
        P_[f"LSTM: {name}"] = np.mean(ps, 0)
        for s, p in zip(SEEDS, ps): P_[f"_LSTM seed {s}: {name}"] = p
        print(f"{target} | LSTM {name}: {time.time()-t0:.0f}s")
    preds[target] = (rows[["ticker", "date", "split"]].assign(y=y), P_)
    for period in ["val", "test_2021", "test_2022_23"]:
        m = sp == period
        sse0 = ((y[m] - ytr_mean) ** 2).sum()
        for name, p in P_.items():
            e = y[m] - p[m]
            results.append({"target": target, "period": period, "model": name, "n": m.sum(),
                            "RMSE": np.sqrt((e ** 2).mean()), "MAE": np.abs(e).mean(), "R2_oos": 1 - (e ** 2).sum() / sse0})
RES = pd.DataFrame(results)
RES.to_csv(os.path.join(DATA_DIR, f"lstm_results_{SENT_VERSION}.csv"), index=False)
show = RES[~RES.model.str.startswith("_")]
for target in TARGETS:
    print("\n=== ", target, "(", TARGETS[target], ")  out-of-sample R2 ===")
    print(show[show.target == target].pivot_table(index="model", columns="period", values="R2_oos").round(3)
          .sort_values("test_2021", ascending=False).to_string())

y_log_park | LSTM market: 51s
y_log_park | LSTM market + news (BiLSTM): 41s
y_log_park | LSTM market + news (LLM): 42s
y_abn_logvol | LSTM market: 39s
y_abn_logvol | LSTM market + news (BiLSTM): 31s
y_abn_logvol | LSTM market + news (LLM): 31s

===  y_log_park ( next-day log range volatility )  out-of-sample R2 ===
period                         test_2021  test_2022_23    val
model                                                        
naive (today)                      0.801         0.655  0.587
ridge: market + news (BiLSTM)      0.779         0.687  0.632
ridge: market + news (LLM)         0.778         0.687  0.633
ridge: market                      0.764         0.681  0.631
LSTM: market + news (LLM)          0.672         0.718  0.632
LSTM: market + news (BiLSTM)       0.671         0.711  0.625
LSTM: market                       0.669         0.713  0.646

===  y_abn_logvol ( next-day abnormal log volume )  out-of-sample R2 ===
period                         test_2021  test_2022

## 3. Is the difference significant? (Diebold–Mariano, Newey–West lag 5)

In [5]:
dm_rows = []
for target in TARGETS:
    meta, P_ = preds[target]
    for period in ["test_2021", "test_2022_23"]:
        m = (meta.split == period).to_numpy(); y = meta.y.to_numpy()[m]
        for a, b in [("LSTM: market + news (BiLSTM)", "LSTM: market"), ("LSTM: market + news (LLM)", "LSTM: market"),
                     ("LSTM: market", "ridge: market"), ("ridge: market + news (BiLSTM)", "ridge: market")]:
            st, p = dm_test(y - P_[a][m], y - P_[b][m])
            dm_rows.append({"target": target, "period": period, "A": a, "B": b, "MSE_A/MSE_B": ((y - P_[a][m]) ** 2).mean() / ((y - P_[b][m]) ** 2).mean(), "DM_stat": st, "p": p})
DM = pd.DataFrame(dm_rows); print(DM.round(3).to_string(index=False))
print("(DM_stat < 0 means A has smaller squared errors than B)")
DM.to_csv(os.path.join(DATA_DIR, f"lstm_dm_tests_{SENT_VERSION}.csv"), index=False)
seed_sd = RES[RES.model.str.startswith("_")].assign(model=lambda d: d.model.str.split(": ").str[1]).groupby(["target", "period", "model"]).R2_oos.agg(["mean", "std"])
print("\nLSTM R2_oos across the 3 seeds:\n", seed_sd.round(3).to_string())

      target       period                             A             B  MSE_A/MSE_B  DM_stat     p
  y_log_park    test_2021  LSTM: market + news (BiLSTM)  LSTM: market        0.992   -0.843 0.399
  y_log_park    test_2021     LSTM: market + news (LLM)  LSTM: market        0.990   -0.850 0.396
  y_log_park    test_2021                  LSTM: market ridge: market        1.407    3.301 0.001
  y_log_park    test_2021 ridge: market + news (BiLSTM) ridge: market        0.937   -7.114 0.000
  y_log_park test_2022_23  LSTM: market + news (BiLSTM)  LSTM: market        1.008    0.769 0.442
  y_log_park test_2022_23     LSTM: market + news (LLM)  LSTM: market        0.982   -2.080 0.038
  y_log_park test_2022_23                  LSTM: market ridge: market        0.899   -3.846 0.000
  y_log_park test_2022_23 ridge: market + news (BiLSTM) ridge: market        0.981   -5.017 0.000
y_abn_logvol    test_2021  LSTM: market + news (BiLSTM)  LSTM: market        0.977   -1.674 0.094
y_abn_logvol    test

In [6]:
# R2 by ticker in the 2021 test year
rows = []
for target in TARGETS:
    meta, P_ = preds[target]
    ytr = meta.y[meta.split == "train"].mean()
    for t in ["GME", "AMC", "BB"]:
        m = ((meta.split == "test_2021") & (meta.ticker == t)).to_numpy(); y = meta.y.to_numpy()[m]
        for name in ["naive (today)", "ridge: market", "LSTM: market", "LSTM: market + news (BiLSTM)", "LSTM: market + news (LLM)"]:
            rows.append({"target": target, "ticker": t, "model": name, "R2_oos": 1 - ((y - P_[name][m]) ** 2).sum() / ((y - ytr) ** 2).sum()})
print(pd.DataFrame(rows).pivot_table(index=["target", "model"], columns="ticker", values="R2_oos").round(3).to_string())

ticker                                       AMC     BB    GME
target       model                                            
y_abn_logvol LSTM: market                  0.476  0.559  0.375
             LSTM: market + news (BiLSTM)  0.481  0.564  0.401
             LSTM: market + news (LLM)     0.469  0.556  0.397
             naive (today)                 0.591  0.683  0.360
             ridge: market                 0.308  0.612 -0.112
y_log_park   LSTM: market                  0.723  0.676  0.613
             LSTM: market + news (BiLSTM)  0.724  0.654  0.628
             LSTM: market + news (LLM)     0.722  0.663  0.627
             naive (today)                 0.830  0.683  0.819
             ridge: market                 0.792  0.688  0.768


## 4. Reddit sentiment in 2021 (pooled regression, ticker fixed effects, SEs clustered by date)
Controls: today's log range vol, 5-day vol, abnormal volume, return, and news sentiment and count. Same sample for every model.

In [7]:
R21 = D[(D.rd_has == 1)].dropna(subset=["y_log_park", "y_abn_logvol", "vol_5", "abn_logvol", "qiss_reddit"]).copy()   # same sample for every model
print("Reddit-covered ticker-days:", len(R21), R21.groupby("ticker").size().to_dict())
reg_rows = []
for target in TARGETS:
    base = ["log_park", "vol_5", "abn_logvol", "ret", "nb_sent", "nb_logn"]
    for name, extra in [("controls only", []), ("+ Reddit sentiment", ["rd_sent", "rd_clar", "rd_amb"]),
                        ("+ Reddit sentiment + attention", ["rd_sent", "rd_clar", "rd_amb", "rd_logn"]), ("+ QISS (Reddit)", ["qiss_reddit"])]:
        X = pd.get_dummies(R21[base + extra + ["ticker"]], columns=["ticker"], drop_first=True, dtype=float)
        X = sm.add_constant(X); y = R21[target]
        fit = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(R21.date)[0]})
        row = {"target": target, "model": name, "n": int(fit.nobs), "R2": fit.rsquared}
        for v in extra: row[f"{v} coef (t)"] = f"{fit.params[v]:.3f} ({fit.tvalues[v]:.1f})"
        reg_rows.append(row)
REG = pd.DataFrame(reg_rows); print(REG.round(3).to_string(index=False))
REG.to_csv(os.path.join(DATA_DIR, f"reddit_2021_regressions_{SENT_VERSION}.csv"), index=False)

Reddit-covered ticker-days: 204 {'AMC': 73, 'BB': 83, 'GME': 48}
      target                          model   n    R2 rd_sent coef (t) rd_clar coef (t) rd_amb coef (t) rd_logn coef (t) qiss_reddit coef (t)
  y_log_park                  controls only 204 0.578              NaN              NaN             NaN              NaN                  NaN
  y_log_park             + Reddit sentiment 204 0.581      0.675 (1.0)    -1.902 (-1.2)     4.223 (1.3)              NaN                  NaN
  y_log_park + Reddit sentiment + attention 204 0.583      0.486 (0.8)    -2.289 (-1.4)     4.439 (1.4)      0.043 (1.2)                  NaN
  y_log_park                + QISS (Reddit) 204 0.578              NaN              NaN             NaN              NaN          0.017 (0.3)
y_abn_logvol                  controls only 204 0.695              NaN              NaN             NaN              NaN                  NaN
y_abn_logvol             + Reddit sentiment 204 0.696      0.189 (0.3)    -0.967 (-